In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_all = pd.read_excel('../data/data_battery_materials_20.06.xlsx')
print(df_all.shape, df_all.columns)
df_all = df_all[['dataset', 'smiles (substrate)', 'capacitance_max']].reset_index()
df_all.rename(columns={'smiles (substrate)': 'smiles', 'capacitance_max': 'capacity_max'}, inplace=True)

sns.histplot(data=df_all, x='capacity_max', bins=50, kde=True)
plt.show()

In [ ]:
df_all = df_all[df_all['capacity_max'] > 0]
df_all = df_all[df_all['capacity_max'] < 1000]
print(df_all.shape)
sns.histplot(data=df_all, x='capacity_max', bins=50, kde=True)
plt.show()

In [ ]:
from rdkit import Chem

rows_to_remove = []
for i, row in df_all.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        rows_to_remove.append(i)
df_all = df_all.drop(rows_to_remove).reset_index(drop=True)
df_all

In [ ]:
from src.core.fingerprints import Fingerprints
df = df_all

names = ['ecfp', 'descriptor']
params = {
    'ecfp': {'radius': 2, 'size': 1024, 'count': True},
    'descriptor': {}
}
fingerprints, f_names = Fingerprints().apply(
    smiles=df['smiles'].tolist(),
    names=names,
    **params
)
df_fingerprints = pd.DataFrame(fingerprints, columns=f_names)
threshold = 0.85
df_fingerprints = df_fingerprints[[col for col in df_fingerprints.columns if df_fingerprints[col].value_counts(normalize=True).iloc[0] <= threshold]]
df_fingerprints = df_fingerprints.loc[:, df_fingerprints.nunique() > 1]
df_fingerprints['capacity_max'] = df['capacity_max'].values
df_fingerprints['smiles'] = df['smiles'].values

df_fingerprints

In [ ]:
names_join = '_'.join(names)
df_fingerprints.to_csv(f'../data/data_batteries_{names_join}.csv', index=False)